In [6]:
import numpy as np
from src.agents import EpsilonGreedyAgent, UCBAgent, SoftmaxAgent
from src.plotting import plot_optimal_selections, plot_regret

def run_experiment(arms, n_steps=1000, n_runs=100):
    """
    Bucle maestro para ejecutar experimentos comparativos.
    :param arms: Lista de objetos Arm (Bernoulli, Binomial o Normal).
    :param n_steps: Horizonte temporal (T).
    :param n_runs: Número de simulaciones para promediar resultados.
    """
    k_arms = len(arms)
    # Identificar cuál es el brazo óptimo (basado en su media real mu)
    optimal_arm_idx = np.argmax([arm.mu for arm in arms])
    
    # Definir los algoritmos a comparar
    agents = [
        EpsilonGreedyAgent(k_arms, epsilon=0.1),
        UCBAgent(k_arms, c=2.0),
        SoftmaxAgent(k_arms, temperature=0.1)
    ]
    
    # Matrices para almacenar resultados (Algoritmos x Pasos)
    # Promediaremos sobre n_runs para que las gráficas no tengan tanto ruido
    avg_regret = np.zeros((len(agents), n_steps))
    avg_optimal_selection = np.zeros((len(agents), n_steps))
    
    for run in range(n_runs):
        for i, agent in enumerate(agents):
            agent.reset() # Resetear estadísticas del agente en cada run
            cumulative_regret = 0
            
            for t in range(n_steps):
                # 1. El agente elige una acción
                action = agent.get_action()
                
                # 2. El entorno (brazo) devuelve una recompensa
                reward = arms[action].draw()
                
                # 3. El agente actualiza su conocimiento
                agent.update(action, reward)
                
                # 4. Cálculo de métricas
                # Regret: mu_optimo - mu_seleccionado
                regret = arms[optimal_arm_idx].mu - arms[action].mu
                cumulative_regret += regret
                avg_regret[i, t] += cumulative_regret
                
                # Selección Óptima: 1 si eligió el mejor, 0 si no
                if action == optimal_arm_idx:
                    avg_optimal_selection[i, t] += 1
                    
    # Promediar resultados por el número de ejecuciones
    avg_regret /= n_runs
    avg_optimal_selection /= n_runs
    
    return agents, avg_regret, avg_optimal_selection

# --- EJEMPLO DE USO EN NOTEBOOK BERNOULLI ---
# from src.arms import BernoulliArm
# arms = [BernoulliArm(0.1), BernoulliArm(0.8), BernoulliArm(0.5)]
# algos, regret, optimal = run_experiment(arms)
# plot_regret(1000, regret, algos)